# Model Publishing (pytorch lightning finetuning)

In [ ]:
HF_TOKEN = ""

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
# Substitua pelo nome do repositório: "sua-organizacao/nome-do-modelo"
repo_id = os.environ.get("HF_MODEL_REPO", "seu-usuario/segmentation-finetuned")
output_dir = "./modelo_hf_3"

# cria privado dentro da org
api.create_repo(repo_id=repo_id, repo_type="model", private=True,token=HF_TOKEN)

In [ ]:
import os
import json
import torch
from pathlib import Path
from safetensors.torch import save_file

# Caminho do checkpoint Lightning (.ckpt)
checkpoint_path = Path(os.environ.get("LOCAL_MODEL_CKPT",
    "lightning_logs/version_0/checkpoints/best.ckpt"))

device = torch.device("cpu")  # carregar em CPU para conversão

# Carregar o checkpoint completo (weights_only=False necessário para .ckpt do Lightning,
# que contém TorchVersion e outros metadados não-tensor no pickle)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

# Extrair APENAS os pesos do modelo (sem optimizer state, hyperparameters, TorchVersion, etc.)
# O HuggingFace Trainer e a API esperam só o state_dict — não o checkpoint completo
state_dict = checkpoint.get("state_dict", checkpoint)

# Remover prefixo "model." adicionado pelo Lightning Module, se existir
state_dict = {
    k.removeprefix("model."): v
    for k, v in state_dict.items()
    if isinstance(v, torch.Tensor)
}

# Diretório de saída
output_dir = "./modelo_hf"
os.makedirs(output_dir, exist_ok=True)

# Salvar como SafeTensors — metadata={"format": "pt"} obrigatório para que
# transformers.from_pretrained() não falhe com AttributeError: 'NoneType'.get
safetensors_path = os.path.join(output_dir, "model.safetensors")
save_file(state_dict, safetensors_path, metadata={"format": "pt"})
print(f"[OK] {safetensors_path}")

# config.json no formato esperado pelo SegmentationModel (diarizers/HuggingFace)
config_json = {
    "architectures": ["SegmentationModel"],
    "chunk_duration": 10,
    "max_speakers_per_chunk": 3,
    "max_speakers_per_frame": 2,
    "min_duration": None,
    "model_type": "pyannet",
    "sample_rate": 16000,
    "torch_dtype": "float32",
    "transformers_version": "4.46.0",
    "warm_up": [0.0, 0.0],
    "weigh_by_cardinality": False,
}
with open(os.path.join(output_dir, "config.json"), "w") as f:
    json.dump(config_json, f, indent=2)
print(f"[OK] config.json")

print("\nArquivos prontos em:", output_dir)
for fname in sorted(os.listdir(output_dir)):
    size = os.path.getsize(os.path.join(output_dir, fname))
    print(f"  {fname}  ({size / 1e6:.1f} MB)" if size > 1000 else f"  {fname}")

In [ ]:
from huggingface_hub import HfApi, login

# API
api = HfApi()

# Upload da pasta inteira
api.upload_folder(
    folder_path=output_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload do modelo fine-tuned",
    token=HF_TOKEN
)

print(f"Modelo upado em: https://huggingface.co/{repo_id}")

In [ ]:
from pyannote.audio import Model

model = Model.from_pretrained(repo_id, use_auth_token=HF_TOKEN, strict=False)

# Model Publishing v2 (Diarizers community finetuning)

In [ ]:
HF_TOKEN = ""

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
# Substitua pelo nome do repositório: "sua-organizacao/nome-do-modelo"
repo_id = os.environ.get("HF_MODEL_REPO", "seu-usuario/segmentation-finetuned")

# cria privado dentro da org
api.create_repo(repo_id=repo_id, repo_type="model", private=True,token=HF_TOKEN)

In [ ]:
from huggingface_hub import HfApi, login

# Aponte para o checkpoint do modelo diarizers que você treinou
output_dir = os.environ.get("LOCAL_MODEL_CKPT",
    "speaker-segmentation-finetuned/checkpoint-best")

# API
api = HfApi()

# Upload da pasta inteira
api.upload_folder(
    folder_path=output_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload do modelo fine-tuned",
    token=HF_TOKEN
)

print(f"Modelo upado em: https://huggingface.co/{repo_id}")

In [ ]:
from diarizers import SegmentationModel

model = SegmentationModel().from_pretrained("", token="")
model = model.to_pyannote_model()